Week 7 Deliverable: Outlier Detection and Data Quality

1. Applies IQR outlier detection to ClosePrice, LivingArea, DaysOnMarket
2. Adds outlier flag columns (does NOT delete records outright)
3. Saves a full flagged dataset and a clean, analysis-ready filtered dataset
4. Prints a before/after comparison of dataset size and median values

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

IQR_FIELDS = ['ClosePrice', 'LivingArea', 'DaysOnMarket']
IQR_MULTIPLIER = 1.5

# PART 1 - SOLD DATASET

In [ ]:
print("=" * 70)
print("OUTLIER DETECTION: SOLD  (sold_features.csv)")
print("=" * 70)

df_sold = pd.read_csv('sold_features.csv', low_memory=False)
rows_before_sold = len(df_sold)
print(f"\nLoaded {rows_before_sold} rows")

In [ ]:
# Median values BEFORE any outlier filtering, for the before/after comparison
medians_before_sold = df_sold[IQR_FIELDS].median()
print(f"\nMedian values before outlier filtering:\n{medians_before_sold}")

In [ ]:
# --- Business rule flags (always invalid, regardless of IQR) ---
# These were already created in Week 4-5 (invalid_close_price_flag etc.),
# combined here into one flag for convenience.
business_rule_flag_sold = pd.Series(False, index=df_sold.index)
if 'invalid_close_price_flag' in df_sold.columns:
    business_rule_flag_sold = business_rule_flag_sold | df_sold['invalid_close_price_flag']
if 'invalid_living_area_flag' in df_sold.columns:
    business_rule_flag_sold = business_rule_flag_sold | df_sold['invalid_living_area_flag']
if 'invalid_days_on_market_flag' in df_sold.columns:
    business_rule_flag_sold = business_rule_flag_sold | df_sold['invalid_days_on_market_flag']

In [ ]:
# --- IQR outlier flags for each field ---
print("\n--- IQR outlier flags ---")
iqr_flag_cols_sold = []
for field in IQR_FIELDS:
    q1 = df_sold[field].quantile(0.25)
    q3 = df_sold[field].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr

    flag_col = f'{field}_outlier_flag'
    df_sold[flag_col] = (df_sold[field] < lower) | (df_sold[field] > upper)
    iqr_flag_cols_sold.append(flag_col)

    print(f"  {field}: Q1={q1:,.2f}, Q3={q3:,.2f}, IQR={iqr:,.2f}, "
          f"bounds=[{lower:,.2f}, {upper:,.2f}], flagged={df_sold[flag_col].sum()}")

In [ ]:
# --- Combined "any outlier" flag ---
df_sold['any_outlier_flag'] = business_rule_flag_sold
for col in iqr_flag_cols_sold:
    df_sold['any_outlier_flag'] = df_sold['any_outlier_flag'] | df_sold[col]

print(f"\nTotal records flagged by business rules (invalid values): {business_rule_flag_sold.sum()}")
print(f"Total records flagged by any IQR rule:                     "
      f"{df_sold[iqr_flag_cols_sold].any(axis=1).sum()}")
print(f"Total records flagged by any_outlier_flag (combined):      {df_sold['any_outlier_flag'].sum()}")

In [ ]:
# --- Save the FULL flagged dataset (nothing removed) ---
df_sold.to_csv('sold_flagged.csv', index=False, encoding='utf-8')
print(f"\nSaved full flagged dataset: sold_flagged.csv ({len(df_sold)} rows, {len(df_sold.columns)} columns)")

In [ ]:
# --- Save a CLEAN, analysis-ready dataset (outliers excluded) ---
df_sold_clean = df_sold[~df_sold['any_outlier_flag']].copy()
rows_after_sold = len(df_sold_clean)
medians_after_sold = df_sold_clean[IQR_FIELDS].median()

df_sold_clean.to_csv('sold_analysis_ready.csv', index=False, encoding='utf-8')
print(f"Saved clean filtered dataset: sold_analysis_ready.csv ({rows_after_sold} rows)")

In [ ]:
# --- Before/After comparison ---
print("\n--- Before/After Comparison (sold) ---")
print(f"Row count: {rows_before_sold} -> {rows_after_sold} "
      f"({rows_before_sold - rows_after_sold} records excluded, "
      f"{(rows_before_sold - rows_after_sold) / rows_before_sold * 100:.1f}%)")

comparison_sold = pd.DataFrame({
    'median_before': medians_before_sold,
    'median_after': medians_after_sold,
})
comparison_sold['pct_change'] = (
    (comparison_sold['median_after'] - comparison_sold['median_before'])
    / comparison_sold['median_before'] * 100
).round(2)
print(comparison_sold)

# PART 2 - LISTING DATASET

In [ ]:
print("=" * 70)
print("OUTLIER DETECTION: LISTING  (listing_features.csv)")
print("=" * 70)

df_listing = pd.read_csv('listing_features.csv', low_memory=False)
rows_before_listing = len(df_listing)
print(f"\nLoaded {rows_before_listing} rows")

In [ ]:
medians_before_listing = df_listing[IQR_FIELDS].median()
print(f"\nMedian values before outlier filtering:\n{medians_before_listing}")

In [ ]:
business_rule_flag_listing = pd.Series(False, index=df_listing.index)
if 'invalid_close_price_flag' in df_listing.columns:
    business_rule_flag_listing = business_rule_flag_listing | df_listing['invalid_close_price_flag']
if 'invalid_living_area_flag' in df_listing.columns:
    business_rule_flag_listing = business_rule_flag_listing | df_listing['invalid_living_area_flag']
if 'invalid_days_on_market_flag' in df_listing.columns:
    business_rule_flag_listing = business_rule_flag_listing | df_listing['invalid_days_on_market_flag']

In [ ]:
print("\n--- IQR outlier flags ---")
iqr_flag_cols_listing = []
for field in IQR_FIELDS:
    q1 = df_listing[field].quantile(0.25)
    q3 = df_listing[field].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr

    flag_col = f'{field}_outlier_flag'
    df_listing[flag_col] = (df_listing[field] < lower) | (df_listing[field] > upper)
    iqr_flag_cols_listing.append(flag_col)

    print(f"  {field}: Q1={q1:,.2f}, Q3={q3:,.2f}, IQR={iqr:,.2f}, "
          f"bounds=[{lower:,.2f}, {upper:,.2f}], flagged={df_listing[flag_col].sum()}")

In [ ]:
df_listing['any_outlier_flag'] = business_rule_flag_listing
for col in iqr_flag_cols_listing:
    df_listing['any_outlier_flag'] = df_listing['any_outlier_flag'] | df_listing[col]

print(f"\nTotal records flagged by business rules (invalid values): {business_rule_flag_listing.sum()}")
print(f"Total records flagged by any IQR rule:                     "
      f"{df_listing[iqr_flag_cols_listing].any(axis=1).sum()}")
print(f"Total records flagged by any_outlier_flag (combined):      {df_listing['any_outlier_flag'].sum()}")

In [ ]:
df_listing.to_csv('listing_flagged.csv', index=False, encoding='utf-8')
print(f"\nSaved full flagged dataset: listing_flagged.csv ({len(df_listing)} rows, {len(df_listing.columns)} columns)")

In [ ]:
df_listing_clean = df_listing[~df_listing['any_outlier_flag']].copy()
rows_after_listing = len(df_listing_clean)
medians_after_listing = df_listing_clean[IQR_FIELDS].median()

df_listing_clean.to_csv('listing_analysis_ready.csv', index=False, encoding='utf-8')
print(f"Saved clean filtered dataset: listing_analysis_ready.csv ({rows_after_listing} rows)")

In [ ]:
print("\n--- Before/After Comparison (listing) ---")
print(f"Row count: {rows_before_listing} -> {rows_after_listing} "
      f"({rows_before_listing - rows_after_listing} records excluded, "
      f"{(rows_before_listing - rows_after_listing) / rows_before_listing * 100:.1f}%)")

comparison_listing = pd.DataFrame({
    'median_before': medians_before_listing,
    'median_after': medians_after_listing,
})
comparison_listing['pct_change'] = (
    (comparison_listing['median_after'] - comparison_listing['median_before'])
    / comparison_listing['median_before'] * 100
).round(2)
print(comparison_listing)